In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# Charger la data originale pour comparer
dataset = fetch_ucirepo(id=544)
df_original = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df = df_original.copy()

# ── AVANT ────────────────────────────────────────────────────────────────────
print("=" * 50)
print("AVANT — Data originale")
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── ENCODAGE ─────────────────────────────────────────────────────────────────
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

le = LabelEncoder()
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    df[col] = le.fit_transform(df[col])

le_target = LabelEncoder()
df['NObeyesdad'] = le_target.fit_transform(df['NObeyesdad'])

# ── APRÈS ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("APRÈS — Data encodée")s
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── MÉMOIRE ───────────────────────────────────────────────────────────────────
def optimize_memory(df):
    before = df.memory_usage(deep=True).sum() / 1024
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    after = df.memory_usage(deep=True).sum() / 1024
    print(f"\n{'='*50}")
    print("OPTIMISATION MÉMOIRE")
    print(f"{'='*50}")
    print(f"Avant  : {before:.1f} KB")
    print(f"Après  : {after:.1f} KB")
    print(f"Gagné  : {((before-after)/before*100):.1f}%")
    return df

df = optimize_memory(df)

# ── SPLIT ─────────────────────────────────────────────────────────────────────
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n{'='*50}")
print("SÉPARATION TRAIN / TEST")
print(f"{'='*50}")
print(f"Total lignes    : {len(df)}")
print(f"Train (80%)     : {X_train.shape[0]} lignes")
print(f"Test  (20%)     : {X_test.shape[0]} lignes")
print(f"\n✅ Data prête pour le ML !")

AVANT — Data originale
   Gender SMOKE       CAEC     NObeyesdad
0  Female    no  Sometimes  Normal_Weight
1  Female   yes  Sometimes  Normal_Weight
2    Male    no  Sometimes  Normal_Weight

Types : 
Gender        str
SMOKE         str
CAEC          str
NObeyesdad    str
dtype: object

APRÈS — Data encodée
   Gender  SMOKE  CAEC  NObeyesdad
0       0      0     2           1
1       0      1     2           1
2       1      0     2           1

Types : 
Gender        int64
SMOKE         int64
CAEC          int64
NObeyesdad    int64
dtype: object

OPTIMISATION MÉMOIRE
Avant  : 280.5 KB
Après  : 140.3 KB
Gagné  : 50.0%

SÉPARATION TRAIN / TEST
Total lignes    : 2111
Train (80%)     : 1688 lignes
Test  (20%)     : 423 lignes

✅ Data prête pour le ML !


In [ ]:
# Sauvegarder la data complète encodée
df.to_csv('data/obesity_prepared.csv', index=False)

# Sauvegarder train et test séparément
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print("✅ Fichiers sauvegardés dans le dossier data/")